# Update Price Notebook
Separates the workflow into steps and calls the SAP utilities from `sap_xml_utils.py`.

In [16]:
import pandas as pd
from decimal import Decimal, InvalidOperation
import importlib

import sap_xml_utils as sxu

sxu = importlib.reload(sxu)

from sap_xml_utils import (
    get_item_id_by_sku,
    get_current_item_price,
    update_item_price as util_update_item_price,
)

## Configuration

In [17]:
INPUT_CSV = 'input_updates.csv'
df = pd.read_csv(INPUT_CSV, dtype=str).fillna('')
df.head()

,sales_order_id,sku_product,new_price
0,699331,C19027LAV,44
1,699331,C19026LAV,45
2,699331,C19025LAV,46
3,679322,C19027LAV,51
4,679322,C19026LAV,52


## Helpers

In [18]:
def _to_decimal(s):
    try:
        return Decimal(str(s))
    except (InvalidOperation, ValueError):
        return None

## Validate columns

In [19]:
required = {'sales_order_id', 'sku_product', 'new_price'}
if not required.issubset(set([c.strip() for c in df.columns.tolist()])):
    raise ValueError('CSV must include: sales_order_id, sku_product, new_price')
df[sorted(required)]

,new_price,sales_order_id,sku_product
0,44,699331,C19027LAV
1,45,699331,C19026LAV
2,46,699331,C19025LAV
3,51,679322,C19027LAV
4,52,679322,C19026LAV
5,53,679322,C19024LAV


## Process rows: read current price, compare, and update when equal

In [24]:
logs = []
for row in df.itertuples(index=False):
    so_id = getattr(row, 'sales_order_id', '').strip()
    sku = getattr(row, 'sku_product', '').strip()
    new_price = getattr(row, 'new_price', '').strip()
    item_id = getattr(row, 'item_id', '').strip() if ('item_id' in df.columns) else ''

    if not item_id:
        item_id = get_item_id_by_sku(so_id, sku)
        if not item_id:
            logs.append({'sales_order_id': so_id, 'sku_product': sku, 'status': 'skip_no_item'})
            continue

    current_price = get_current_item_price(so_id, item_id)
    d_curr = _to_decimal(current_price)
    d_new = _to_decimal(new_price)
    if d_curr is None or d_new is None:
        logs.append({'sales_order_id': so_id, 'sku_product': sku, 'item_id': item_id, 'status': 'skip_bad_price', 'current_price': current_price, 'new_price': new_price})
        continue

    if d_curr == d_new:
        resp = util_update_item_price(so_id, item_id, new_price)
        logs.append({'sales_order_id': so_id, 'sku_product': sku, 'item_id': item_id, 'action': 'updated', 'response_len': len(resp)})
    else:
        logs.append({'sales_order_id': so_id, 'sku_product': sku, 'item_id': item_id, 'action': 'no_update', 'current_price': str(d_curr), 'new_price': str(d_new)})

pd.DataFrame(logs)

Exception: SOAP call failed